<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/05_Taller_Embeddings_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: embeddings, RNN, LSTM y GRU</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica los temas de `11_Embeddings_Redes_Neuronales.ipynb` y `13_RNN_LSTM_GRU_PLN.ipynb` con **una tarea distinta**: en lugar de clasificar las reseñas de Amazon en positivas y negativas, predeciremos la **calificación completa de 1 a 5 estrellas**. Es un problema más difícil y con una particularidad: las clases están **ordenadas** (confundir 4 con 5 estrellas es menos grave que confundir 1 con 5).

Enfocamos:

- decisiones de vectorización justificadas con datos (longitud de secuencia y tamaño del vocabulario),
- clasificación multiclase frente a **regresión ordinal**, con métricas adecuadas,
- comparación de arquitecturas recurrentes con máscaras,
- el valor de los **embeddings preentrenados** según la cantidad de datos etiquetados,
- el análisis de errores en un problema ordinal.

Los modelos son pequeños para que el taller se pueda ejecutar en CPU (o en Colab). Resuelve cada ejercicio en la celda indicada y responde las preguntas en celdas markdown.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Decisiones de vectorización</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. Clasificación frente a regresión ordinal</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Arquitecturas recurrentes</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Embeddings preentrenados y cantidad de datos</span>](#ejercicio-4)
- [<span style="color: #005C99;">Ejercicio 5. Análisis de errores en un problema ordinal</span>](#ejercicio-5)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scikit-learn tensorflow gensim datasets

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # oculta mensajes informativos de TensorFlow

import re
import string
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # fija las semillas de Python, NumPy y TensorFlow

from gensim.models import Word2Vec
from datasets import load_dataset
from sklearn.metrics import accuracy_score, mean_absolute_error, ConfusionMatrixDisplay

resenas = load_dataset("SetFit/amazon_reviews_multi_es")
df_train = resenas["train"].to_pandas().sample(n=50_000, random_state=SEED)
df_val = resenas["validation"].to_pandas()
df_test = resenas["test"].to_pandas()

X_train_txt, y_train = df_train.text.tolist(), df_train.label.to_numpy()   # etiquetas 0..4 = 1..5 estrellas
X_val_txt, y_val = df_val.text.tolist(), df_val.label.to_numpy()
X_test_txt, y_test = df_test.text.tolist(), df_test.label.to_numpy()
print(f"Entrenamiento: {len(X_train_txt)} | Validación: {len(X_val_txt)} | Prueba: {len(X_test_txt)}")
print("Distribución de estrellas en entrenamiento:", (pd.Series(y_train + 1).value_counts(normalize=True).sort_index().round(3)).to_dict())

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Decisiones de vectorización</h2>

**Objetivo:** elegir la longitud de las secuencias y el tamaño del vocabulario a partir de los datos, no a ojo.

1. Calcula la distribución del número de palabras por reseña en entrenamiento (usa la función `estandarizar` dada y `split()`). Reporta los percentiles 50, 90, 95 y 99 por número de estrellas. ¿Las reseñas negativas son más largas o más cortas?
2. Elige `LONGITUD` como el percentil 95 redondeado a la decena superior. ¿Qué fracción de reseñas se recorta?
3. Cuenta la frecuencia de cada palabra en entrenamiento y grafica la **cobertura**: qué fracción de los tokens del conjunto de validación queda dentro del vocabulario si se usan las $V$ palabras más frecuentes, para $V$ entre 1 000 y 50 000. Elige `MAX_TOKENS` como el menor $V$ que cubre al menos el 97 % de los tokens.
4. Crea la capa `TextVectorization` con esas decisiones y vectoriza los tres conjuntos (`X_train`, `X_val`, `X_test`).

In [ ]:
def estandarizar(texto):
    """Minúsculas y eliminación de puntuación, incluidos los signos de apertura del español."""
    texto = tf.strings.lower(texto)
    return tf.strings.regex_replace(texto, r"[¡¿«»“”\"#$%&()*+,\-./:;<=>?@\[\\\]^_`{|}~!']", " ")


# Escribe tu código aquí
LONGITUD = None
MAX_TOKENS = None

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Clasificación frente a regresión ordinal</h2>

**Objetivo:** comparar dos formas de modelar una etiqueta ordenada y evaluarlas con métricas adecuadas.

Usa la arquitectura de promedio de embeddings del cuaderno 11 (`Embedding(MAX_TOKENS, 64, mask_zero=True)` → `GlobalAveragePooling1D` → `Dense(32, relu)`), con dos cabezas distintas:

- **Clasificación:** `Dense(5, softmax)` con `sparse_categorical_crossentropy`. Predicción: la clase más probable.
- **Regresión:** `Dense(1)` lineal con error cuadrático medio sobre la etiqueta 0–4. Predicción: el valor redondeado y recortado al rango [0, 4].

1. Entrena ambos modelos (máximo 10 épocas, parada temprana con `patience=2` sobre la pérdida de validación).
2. Implementa `metricas(y_real, y_pred)` que devuelva: **exactitud**, **exactitud ±1** (fracción de predicciones a una estrella o menos de la real) y **error absoluto medio** (MAE) en estrellas.
3. Evalúa ambos en **validación**. Agrega una tercera variante: la clasificación, pero prediciendo el **valor esperado** $\sum_k k\, P(k)$ redondeado en lugar de la clase más probable.
4. ¿Qué modelo elegirías si la métrica principal fuera la exactitud? ¿Y si fuera el MAE? Explica la diferencia.

In [ ]:
# Escribe tu código aquí
def metricas(y_real, y_pred):
    pass

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Arquitecturas recurrentes</h2>

**Objetivo:** comparar modelos recurrentes en la tarea de 5 clases, midiendo también su costo.

1. Con la cabeza de clasificación (`Dense(5, softmax)`), entrena y compara:
   - el promedio de embeddings del ejercicio 2,
   - `GRU(64)`,
   - `Bidirectional(GRU(64))`,
   - dos GRU apiladas: `GRU(64, return_sequences=True)` → `GRU(64)`.

   Usa `Embedding(MAX_TOKENS, 64, mask_zero=True)`, `Dropout(0.3)` antes de la capa de salida, máximo 6 épocas y parada temprana con `patience=1`.
2. Reporta en una tabla: número de parámetros, épocas entrenadas, tiempo de entrenamiento y las métricas del ejercicio 2 en **validación**.
3. ¿Justifica la mejora el costo adicional? ¿Qué modelo elegirías para producción y por qué?

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Embeddings preentrenados y cantidad de datos</h2>

**Objetivo:** medir cuánto aportan los embeddings Word2Vec preentrenados según el número de ejemplos etiquetados.

1. Entrena un modelo Word2Vec con gensim (skip-gram, 64 dimensiones, ventana 5, `min_count=3`, 5 épocas) sobre los textos de **las 200 000 reseñas de entrenamiento** (`resenas["train"]["text"]`), tokenizados con la función `palabras` del ejercicio 1. Es texto **sin usar sus etiquetas**.
2. Construye la matriz de embeddings inicial para el vocabulario de `vectorizador` (ceros para las palabras sin vector). ¿Qué fracción del vocabulario tiene vector?
3. Para $n \in \{500, 2\,000, 8\,000\}$ reseñas etiquetadas (submuestras de `X_train`), entrena el modelo de **promedio** de embeddings con 5 clases en dos versiones: embeddings **aleatorios** y embeddings **inicializados con Word2Vec y ajustables**. Usa hasta 30 épocas con parada temprana (`patience=3`) y lotes de 32.
4. Grafica la exactitud en validación contra $n$ (escala log) para ambas versiones e interpreta.

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Análisis de errores en un problema ordinal</h2>

**Objetivo:** entender qué confusiones comete el mejor modelo y cuánto se deben a la ambigüedad de las propias etiquetas.

1. Elige el mejor modelo de los ejercicios 2–3 según **validación** y evalúalo **una vez** en **prueba** con las métricas del ejercicio 2.
2. Muestra su matriz de confusión normalizada por fila. ¿Dónde se concentran los errores?
3. Calcula el *recall* por clase. ¿Qué clase es la más difícil y por qué crees que ocurre?
4. Muestra 6 reseñas de prueba en las que el error es de 3 o más estrellas. Léelas y clasifica la causa probable: error del modelo, etiqueta inconsistente con el texto, ironía, reseña sobre el envío y no sobre el producto, etc.

In [ ]:
# Escribe tu código aquí

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Chollet, F. (2021). *Deep Learning with Python* (2.ª ed.). Manning.
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Keung, P., Lu, Y., Szarvas, G. & Smith, N. A. (2020). The Multilingual Amazon Reviews Corpus. *EMNLP*.